<link rel="stylesheet" href="berkeley.css">

<h1 class="cal cal-h1">Lecture 02: KNN, ML Vocabulary, and K-Means &ndash; CS 189, Fall 2026</h1>


In this notebook we build our way from raw data to our first machine learning model, and then to our
first unsupervised model. We start with the tools we need to look at data (`pandas`, `numpy`, and
plotting), then introduce the simplest model we can think of, **k-nearest neighbors**. Along the way
KNN will force us to invent the ideas of **generalization**, the **train/test split**, and
**hyperparameters**. We close with **k-means**, which solves a related problem without any labels
at all.

The main body of this notebook is what we walk through in lecture. The **Appendix** at the end goes
much deeper on `pandas`, `numpy`, and visualization syntax; work through it after lecture.



In [ ]:
# Download data & Install Dependencies
import os
import requests

os.makedirs("data", exist_ok=True)
data_files = {
    "data/penguins.csv":
        "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv",
}
for path, url in data_files.items():
    if not os.path.exists(path):
        r = requests.get(url)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"Downloaded {path}")
    else:
        print(f"Found {path}")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

px.defaults.width = 800
pd.set_option("plotting.backend", "plotly")

os.makedirs("images", exist_ok=True)

<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 1: Looking at the Data</h2>

Before any modeling, look at the data. This section is a fast tour of the tools; the Appendix has
the full treatment of every function used here.

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">The Palmer Penguins Dataset</h3>


Measurements of 344 penguins from three islands in the Palmer Archipelago, Antarctica. Each row is
one penguin. We will use this single dataset for everything today.



In [ ]:
penguins = pd.read_csv("data/penguins.csv")
penguins.head()

In [ ]:
penguins.info()

In [ ]:
# describe() summarizes the numeric columns
penguins.describe()

How much data do we have, and what are we looking at?

- `shape` gives (rows, columns)
- `value_counts()` counts the occurrences of each value in a column


In [ ]:
print("shape:", penguins.shape)

In [ ]:
penguins["species"].value_counts()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Selecting Data: `loc` and `iloc`</h3>


Two ways to pull a subset out of a `DataFrame`. The distinction matters constantly, so it is worth
getting straight now.

- `iloc[]` selects by **integer position**, like indexing a numpy array. The end of a slice is *excluded*.
- `loc[]` selects by **label**, meaning the index value and the column name. The end of a slice is *included*.



In [ ]:
# iloc: by position. Rows 0-4, first three columns.
penguins.iloc[0:5, 0:3]

In [ ]:
# loc: by label. Rows 0-4 (inclusive!), named columns.
penguins.loc[0:4, ["species", "island", "bill_length_mm"]]

In [ ]:
# A single column is a Series; a list of columns is a DataFrame
print(type(penguins["bill_length_mm"]))
print(type(penguins[["bill_length_mm"]]))

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Filtering and Missing Values</h3>


Real data has missing information. `isna()` finds them and `dropna()` removes the rows with missing `NULL` values.



In [ ]:
penguins.isna().sum()

In [ ]:
# Boolean filtering: which penguins are both heavy and long-flippered?
mask = (penguins["body_mass_g"] > 5000) & (penguins["flipper_length_mm"] > 220)
penguins[mask].head()

In [ ]:
df = penguins.dropna()
print(len(penguins), "rows ->", len(df), "rows after dropping missing values")
df.head()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Grouping and Summarizing</h3>


`groupby()` splits the data into groups, applies a function to each, and combines the results.



In [ ]:
df.groupby("species")[["bill_length_mm", "flipper_length_mm", "body_mass_g"]].mean().round(1)

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">`numpy`: the Array Underneath</h3>


`pandas` is built on `numpy`. Models in `scikit-learn` want numpy arrays, so this is how we hand our
data over.



In [ ]:
FEATURES = ["bill_length_mm", "flipper_length_mm"]
X = df[FEATURES].to_numpy()      # feature matrix, shape (n_samples, n_features)
y = df["species"].to_numpy()     # labels, shape (n_samples,)

print("X shape:", X.shape, "| y shape:", y.shape)
print("X dtype:", X.dtype, "\n")
print("The first three rows of X:\n", X[:3])

In [ ]:
# Vectorized operations act on the whole array at once, with no Python loop.
print("column means:", X.mean(axis=0).round(2))
print("column stds: ", X.std(axis=0).round(2))

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Visualizing: Can We See the Species?</h3>

If the species separate visually in these two dimensions, then a model has a chance.



In [ ]:
fig = px.scatter(
    df, x="bill_length_mm", y="flipper_length_mm", color="species",
    title="Palmer Penguins: bill length vs flipper length",
    labels={
        "bill_length_mm": "Bill length (mm)",
        "flipper_length_mm": "Flipper length (mm)",
        "species": "Species",
    },
    height=520,
)
# Plotly's default legend sits off to the right and often gets clipped in notebooks.
fig.update_layout(
    showlegend=True,
    legend=dict(
        title_text="Species",
        x=0.01,
        y=0.99,
        xanchor="left",
        yanchor="top",
        bgcolor="white",
        bordercolor="lightgray",
        borderwidth=1,
    ),
)
fig.show()

**Look at what this plot is telling us.** The three species occupy different regions. Nothing
separates them perfectly, but points near each other tend to share a species.

<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 2: The Simplest Possible Model</h2>


We want to predict a penguin's species from its bill length and flipper length. We ask ourselves, what is the least clever thing that could possibly work?



<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">K-Nearest Neighbors (KNN) with k = 1</h3>


**To predict the species of a new penguin, find the penguin in our data that is closest to it, and copy that penguin's species.**

That is the entire algorithm. Notice what it does *not* have any optimization. "Training" is nothing more than storing the data.



In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X, y)          # "training": store the data
print("Model is trained.")

In [ ]:
# Inference: predict the species of a new penguin
new_penguin = np.array([[45.0, 200.0]])   # bill 45mm, flipper 200mm
print("Predicted species:", knn1.predict(new_penguin)[0])

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">How Good Is It?</h3>

We ask the model to predict the species of every penguin in our dataset, and count how often it is right.

In [ ]:
accuracy = knn1.score(X, y)
print(f"Accuracy on our data: {accuracy:.1%}")

### 100%.

We just built a perfect classifier out of the dumbest idea.

**Something is wrong.** What is it?

In [ ]:
# Why 100%? Ask which point is nearest to the first penguin in our data.
distances, indices = knn1.kneighbors(X[:1], n_neighbors=1)
print("Nearest neighbour of penguin 0 is penguin index:", indices[0][0])
print("at a distance of:", distances[0][0])

The nearest neighbour of every penguin in our dataset **is itself**, at distance zero. We asked the model to answer questions it had already memorized the answers to.

This tells us nothing about whether the model has *learned* anything. We need to evaluate it on penguins it has never seen.

<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 3: Generalization and the Train/Test Split</h2>


**Generalization** is the ability to perform well on new, unseen data drawn from the same
distribution. To measure it, we hold data back.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} penguins")
print(f"Test set:     {X_test.shape[0]} penguins")

In [ ]:
knn1 = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)

print(f"Training accuracy: {knn1.score(X_train, y_train):.1%}")
print(f"Test accuracy:     {knn1.score(X_test, y_test):.1%}")

Training accuracy is still 100%, and it always will be for $k=1$, because every training point is still its own nearest neighbour. The test accuracy is the honest number, and it is meaningfully lower.

The gap between those two numbers is the thing we spend the rest of the semester managing.

<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 4: k Is a Choice</h2>


Why look at only one neighbour? Let each of the $k$ nearest neighbours vote, and take the majority.


<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Seeing k in the Decision Boundary</h3>


A **decision boundary** shows what the model would predict at every point in the feature space. It makes the effect of $k$ visible.



In [ ]:
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder

def decision_boundary_figure(k, X_tr, y_tr, resolution=250):
    le = LabelEncoder().fit(y_tr)
    model = KNeighborsClassifier(n_neighbors=k).fit(X_tr, le.transform(y_tr))

    pad = 1.0
    xs = np.linspace(X_tr[:, 0].min() - pad, X_tr[:, 0].max() + pad, resolution)
    ys = np.linspace(X_tr[:, 1].min() - pad, X_tr[:, 1].max() + pad, resolution)
    xx, yy = np.meshgrid(xs, ys)
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig = go.Figure()
    fig.add_trace(go.Heatmap(x=xs, y=ys, z=zz, showscale=False, opacity=0.30,
                             colorscale=[[0, "#002675"], [0.5, "#FDB515"], [1, "#028842"]]))
    for i, cls in enumerate(le.classes_):
        m = y_tr == cls
        fig.add_trace(go.Scatter(
            x=X_tr[m, 0], y=X_tr[m, 1], mode="markers", name=cls,
            marker=dict(size=7, line=dict(width=1, color="white"),
                        color=["#002675", "#FDB515", "#028842"][i])))
    fig.update_layout(title=f"KNN decision boundary, k = {k}",
                      xaxis_title="Bill length (mm)", yaxis_title="Flipper length (mm)",
                      width=800, height=520)
    return fig

decision_boundary_figure(1, X_train, y_train).show()

**k = 1 gives a noisy boundary** with little islands around individual points. The model is contorting itself to get every single training penguin right, including the ones that are unusual.
That is **overfitting**: fitting the noise as well as the signal.

In [ ]:
decision_boundary_figure(15, X_train, y_train).show()

In [ ]:
decision_boundary_figure(100, X_train, y_train).show()

**k = 15** smooths the boundary into something that looks like a real trend.

**k = 100** smooths it so much that the model is barely paying attention to the data. That is
**underfitting**.

So $k$ controls a trade-off, and somewhere in between is a good value.


<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Finding a Good k</h3>


We cannot use the test set to pick $k$. The moment we tune anything against the test set, it stops measuring generalization and becomes just another training set.

So we split again: a **validation set**, carved out of the training data.


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)
print(f"train {len(X_tr)} | validation {len(X_val)} | test {len(X_test)}")

In [ ]:
ks = list(range(1, 101))
train_acc, val_acc = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    train_acc.append(m.score(X_tr, y_tr))
    val_acc.append(m.score(X_val, y_val))

best_k = ks[int(np.argmax(val_acc))]
print(f"Best k on the validation set: {best_k}  (validation accuracy {max(val_acc):.1%})")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ks, y=train_acc, name="Training accuracy",
                         line=dict(color="#002675", width=3)))
fig.add_trace(go.Scatter(x=ks, y=val_acc, name="Validation accuracy",
                         line=dict(color="#FDB515", width=3)))
fig.add_vline(x=best_k, line_dash="dash", line_color="#028842",
              annotation_text=f"best k = {best_k}")
fig.update_layout(title="Training and validation accuracy as k increases",
                  xaxis_title="k (number of neighbours)", yaxis_title="Accuracy",
                  width=800, height=520)
fig.show()

Read this plot carefully, because its shape recurs all semester.

- On the **left** (small $k$) training accuracy is perfect and validation accuracy is lower. Overfitting.
- On the **right** (large $k$) both accuracies fall together. Underfitting.
- The **sweet spot** is where validation accuracy peaks.

$k$ is a **hyperparameter**: a value we choose *before* training, as opposed to a **parameter**, which is learned *from* the data during training. KNN is unusual in having no parameters at all, only this one hyperparameter. That makes it a clean place to see the distinction.


In [ ]:
# Only now, having chosen k, do we touch the test set. Once.
final = KNeighborsClassifier(n_neighbors=best_k).fit(X_train, y_train)
print(f"Final test accuracy with k={best_k}: {final.score(X_test, y_test):.1%}")

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A Wrinkle: Distance Depends on Units</h3>


KNN is built entirely on distance, so it cares about the scale of the features. Flipper length spans roughly 172-231 mm and bill length roughly 32-60 mm, so flipper length contributes far more to the distance simply because its numbers are bigger.



In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
scaled.fit(X_train, y_train)

print(f"Unscaled test accuracy: {final.score(X_test, y_test):.1%}")
print(f"Scaled test accuracy:   {scaled.score(X_test, y_test):.1%}")

**Standardization** rescales each feature to have mean 0 and standard deviation 1, so every
feature contributes to the distance on equal terms. For any distance-based model this is not
optional.


<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 5: The Same Idea for Regression</h2>


So far we predicted a **category** (species). What if we want to predict a **number**, like body mass?

The algorithm barely changes. Find the k nearest neighbours, and instead of taking a majority vote, take their **average**.



In [ ]:
from sklearn.neighbors import KNeighborsRegressor

Xr = df[["flipper_length_mm"]].to_numpy()
yr = df["body_mass_g"].to_numpy()

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
print("Predicting body mass (g) from flipper length (mm)")

In [ ]:
grid = np.linspace(Xr.min(), Xr.max(), 400).reshape(-1, 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=Xr_train.ravel(), y=yr_train, mode="markers", name="Training data",
                         marker=dict(color="lightgray", size=6)))
for k, colour in [(1, "#002675"), (25, "#FDB515"), (200, "#028842")]:
    reg = KNeighborsRegressor(n_neighbors=k).fit(Xr_train, yr_train)
    fig.add_trace(go.Scatter(x=grid.ravel(), y=reg.predict(grid), mode="lines",
                             name=f"k = {k}", line=dict(width=3, color=colour)))
fig.update_layout(title="KNN regression: body mass vs flipper length",
                  xaxis_title="Flipper length (mm)", yaxis_title="Body mass (g)",
                  width=800, height=520)
fig.show()

The **same trade-off**, in a different costume. $k=1$ is a jagged step function chasing every point. $k=200$ is nearly a flat line. The middle is where the real trend lives.

Overfitting and underfitting are not facts about classification. They are facts about model complexity.


In [ ]:
for k in [1, 25, 200]:
    reg = KNeighborsRegressor(n_neighbors=k).fit(Xr_train, yr_train)
    print(f"k={k:>3}  train R^2 = {reg.score(Xr_train, yr_train):.3f}   "
          f"test R^2 = {reg.score(Xr_test, yr_test):.3f}")

At **k=1** the training score is far above the test score: the model is memorizing. 
At **k=200** the two scores collapse *together*, and both are bad: the model is
too simple to capture the trend. The same story the classification curve told, told again.


<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 6: What If We Had No Labels?</h2>


Everything so far was **supervised**: every penguin came with its species attached.

Now suppose a field researcher hands us the same measurements with **no species labels at all**, and
asks: are there natural groups in here?

This is **unsupervised learning**, and specifically **clustering**.



In [ ]:
# Throw the labels away.
fig = px.scatter(df, x="bill_length_mm", y="flipper_length_mm",
                 title="The same penguins, with no labels",
                 labels={"bill_length_mm": "Bill length (mm)",
                         "flipper_length_mm": "Flipper length (mm)"},
                 height=520)
fig.update_traces(marker=dict(color="gray", size=7))
fig.show()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">K-Means in scikit-learn</h3>


K-means partitions the data into K clusters, each represented by a **centroid**. It assigns each
point to the nearest centroid, then moves each centroid to the mean of its points, and repeats.



In [ ]:
from sklearn.cluster import KMeans

Xs = StandardScaler().fit_transform(X)   # distance-based again, so standardize

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster = kmeans.fit_predict(Xs)

df_c = df.copy()
df_c["cluster"] = cluster.astype(str)
px.scatter(df_c, x="bill_length_mm", y="flipper_length_mm", color="cluster",
           title="K-means with K = 3",
           labels={"bill_length_mm": "Bill length (mm)",
                   "flipper_length_mm": "Flipper length (mm)"},
           height=520).show()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Lloyd's Algorithm, Step by Step</h3>


Watch the centroids move. Each iteration is two steps: assign, then update.



In [ ]:
def lloyd_steps(Xs, K=3, n_iter=6, seed=1):
    rng = np.random.default_rng(seed)
    centres = Xs[rng.choice(len(Xs), K, replace=False)].copy()
    history = []
    for _ in range(n_iter):
        d = ((Xs[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
        assign = d.argmin(axis=1)                       # assignment step
        history.append((centres.copy(), assign.copy()))
        for j in range(K):                              # update step
            if (assign == j).any():
                centres[j] = Xs[assign == j].mean(axis=0)
    return history

history = lloyd_steps(Xs)
print(f"{len(history)} iterations recorded")

In [ ]:
palette = ["#002675", "#FDB515", "#028842"]
fig = go.Figure()
frames = []
for step, (centres, assign) in enumerate(history):
    data = []
    for j in range(3):
        m = assign == j
        data.append(go.Scatter(x=Xs[m, 0], y=Xs[m, 1], mode="markers",
                               marker=dict(color=palette[j], size=6), name=f"cluster {j}"))
    data.append(go.Scatter(x=centres[:, 0], y=centres[:, 1], mode="markers",
                           marker=dict(color="black", size=18, symbol="x"), name="centroids"))
    frames.append(go.Frame(data=data, name=str(step)))

fig.add_traces(frames[0].data)
fig.frames = frames
fig.update_layout(
    title="Lloyd's algorithm: assign, then update",
    xaxis_title="Bill length (standardized)", yaxis_title="Flipper length (standardized)",
    width=800, height=560,
    updatemenus=[dict(type="buttons", showactive=False,
                      buttons=[dict(label="Play", method="animate",
                                    args=[None, dict(frame=dict(duration=900, redraw=True),
                                                     fromcurrent=True)])])])
fig.show()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">Choosing K</h3>


K is a hyperparameter, exactly like k in KNN. But there is no validation accuracy to optimize,
because there are no labels. The common heuristic is the **elbow method**: plot the within-cluster
sum of squares (inertia) against K and look for the bend.



In [ ]:
Ks = range(1, 11)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).inertia_ for k in Ks]

fig = px.line(x=list(Ks), y=inertia, markers=True,
              title="Elbow method: inertia vs K",
              labels={"x": "K (number of clusters)", "y": "Within-cluster sum of squares"})
fig.update_traces(line=dict(color="#002675", width=3))
fig.update_layout(width=800, height=480)
fig.show()

The bend is around K=3, which is encouraging. But note how much judgment that reading takes.
The elbow is a heuristic, not a criterion.


<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Part 7: Clusters Are Not Labels</h2>


We have run two models on the same two columns. One was told the species; one was not. How much did
the labels actually buy us?



In [ ]:
comparison = pd.crosstab(df_c["cluster"], df_c["species"])
comparison

In [ ]:
fig = px.scatter(df_c, x="bill_length_mm", y="flipper_length_mm",
                 color="species", symbol="cluster",
                 title="True species (colour) vs discovered clusters (symbol)",
                 labels={"bill_length_mm": "Bill length (mm)",
                         "flipper_length_mm": "Flipper length (mm)"},
                 height=560)
fig.show()

K-means, with no access to the labels at all, largely rediscovered the species.

**But be careful about what that means.** The clusters are not species. K-means found groups of
penguins that are close together in bill and flipper measurements, and in this dataset those groups
happen to line up with species. Change the features and the clusters change. Nothing in the
algorithm knows what a species is, and nothing guarantees the groups it finds correspond to anything
you care about.

<!-- [SLIDO POLL 4 GOES HERE] -->


<link rel="stylesheet" href="berkeley.css">

<h2 class="cal cal-h2">Appendix: Going Deeper</h2>


The material below was not covered in lecture. Work through it on your own; it is the reference for the syntax used above and for Homework 1.



<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A1. `pandas` Data Structures</h3>


A `DataFrame` is a 2-dimensional table. A `Series` is a single column. Both are built on an `Index`.



In [ ]:
s = pd.Series([3750, 3800, 3250], index=["p0", "p1", "p2"], name="body_mass_g")
print(s)
print("\nindex:", s.index.tolist())
print("values:", s.values)

In [ ]:
frame = pd.DataFrame({
    "species": ["Adelie", "Gentoo", "Chinstrap"],
    "bill_length_mm": [39.1, 46.1, 46.5],
    "island": ["Torgersen", "Biscoe", "Dream"],
})
frame

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A2. Exploring a `DataFrame`</h3>


In [ ]:
print(penguins.head(3))          # first rows
print(penguins.tail(3))          # last rows
print(penguins.sample(3))        # random rows
print(penguins.columns.tolist()) # column names
print(penguins.dtypes)           # column types
print(penguins["island"].unique())

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A3. `loc` and `iloc` in Full</h3>


The rule to remember: `iloc` is positional and excludes the endpoint, `loc` is label-based and
includes it.



In [ ]:
print(penguins.iloc[0])              # row 0 as a Series
print(penguins.iloc[0:3])            # rows 0,1,2
print(penguins.iloc[:, 2])           # column at position 2
print(penguins.iloc[[0, 5, 10], [0, 2]])   # arbitrary rows and columns

In [ ]:
print(penguins.loc[0:2])                          # rows labelled 0,1,2 (inclusive)
print(penguins.loc[:, "species":"bill_length_mm"]) # column slice by name
print(penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"].mean())

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A4. Modifying and Sorting</h3>


In [ ]:
tmp = df.copy()
tmp["mass_kg"] = tmp["body_mass_g"] / 1000              # add a column
tmp["bill_ratio"] = tmp["bill_length_mm"] / tmp["bill_depth_mm"]
tmp = tmp.drop(columns=["bill_ratio"])                  # drop a column
tmp.sort_values("body_mass_g", ascending=False).head()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A5. Aggregation, `groupby`, and Pivot Tables</h3>


In [ ]:
print(df.groupby("species")["body_mass_g"].agg(["mean", "std", "count"]).round(1))
print()
print(df.groupby(["species", "island"])["flipper_length_mm"].mean().round(1))
print()
print(df.pivot_table(index="species", columns="island",
                     values="body_mass_g", aggfunc="mean").round(0))

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A6. Joining `DataFrames`</h3>


In [ ]:
islands = pd.DataFrame({
    "island": ["Torgersen", "Biscoe", "Dream"],
    "latitude": [-64.77, -65.43, -64.73],
})
merged = df.merge(islands, on="island", how="left")   # try how="inner" and how="outer"
merged[["species", "island", "latitude"]].head()

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A7. `numpy` Essentials</h3>


In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print("shape:", a.shape, "| ndim:", a.ndim)
print("row sums:", a.sum(axis=1))
print("col means:", a.mean(axis=0))
print("boolean mask:", a[a > 6])
print("broadcasting:", (a - a.mean(axis=0)).round(2))

In [ ]:
# Euclidean distance by hand, which is exactly what KNN computes
p, q = X[0], X[1]
print("manual :", np.sqrt(((p - q) ** 2).sum()))
print("numpy  :", np.linalg.norm(p - q))

<link rel="stylesheet" href="berkeley.css">

<h3 class="cal cal-h3">A8. Visualization Reference</h3>


`matplotlib` and `seaborn` are the static plotting standards; `plotly` gives interactive figures,
which is why we use it in lecture.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["body_mass_g"], bins=25, color="#002675")
axes[0].set_title("Body mass"); axes[0].set_xlabel("g")
sns.scatterplot(data=df, x="bill_length_mm", y="flipper_length_mm",
                hue="species", ax=axes[1])
axes[1].set_title("By species")
plt.tight_layout(); plt.show()

In [ ]:
# Plotly: histogram, box plot, and a faceted scatter
px.histogram(df, x="body_mass_g", color="species", nbins=30, height=420).show()
px.box(df, x="species", y="flipper_length_mm", color="species", height=420).show()
px.scatter(df, x="bill_length_mm", y="flipper_length_mm",
           color="species", facet_col="island", height=420).show()